In [14]:
import pandas as pd
import re

df = pd.read_csv("../data/raw/customer_support_ticket.csv")

In [15]:
df.head(3)

,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,...,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,Urgent,Open,Email,...,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Urgent,Closed,Email,...,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business
2,3,William Anderson,william.anderson651@outlook.com,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Chat,...,2022-12-05,Yes,Yes,Windows,Safari,Bank Transfer,French,Morning,4,Corporate


In [16]:
columns_to_drop = [
    "ticket_id",
    "customer_name",
    "customer_email",
]

df = df.drop(columns_to_drop, axis=1)

df.head()

,product,category,issue_description,resolution_notes,priority,status,channel,region,customer_age,customer_gender,...,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,Urgent,Open,Email,North America,31,Male,...,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Urgent,Closed,Email,South America,23,Other,...,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business
2,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Chat,Europe,47,Female,...,2022-12-05,Yes,Yes,Windows,Safari,Bank Transfer,French,Morning,4,Corporate
3,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my reque...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Social Media,Asia,65,Other,...,2024-04-04,Yes,No,Windows,Chrome,Credit Card,Spanish,Afternoon,7,Corporate
4,Web Portal,Feature Request,The system is not syncing data across devices ...,We have reset the account credentials and advi...,High,Pending Customer,Email,Asia,45,Male,...,2024-08-24,Yes,No,Linux,NaN,Debit Card,Spanish,Evening,3,Corporate


In [17]:
df["browser"] = df["browser"].fillna("Unknown")
df.head()

,product,category,issue_description,resolution_notes,priority,status,channel,region,customer_age,customer_gender,...,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,Urgent,Open,Email,North America,31,Male,...,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Urgent,Closed,Email,South America,23,Other,...,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business
2,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Chat,Europe,47,Female,...,2022-12-05,Yes,Yes,Windows,Safari,Bank Transfer,French,Morning,4,Corporate
3,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my reque...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Social Media,Asia,65,Other,...,2024-04-04,Yes,No,Windows,Chrome,Credit Card,Spanish,Afternoon,7,Corporate
4,Web Portal,Feature Request,The system is not syncing data across devices ...,We have reset the account credentials and advi...,High,Pending Customer,Email,Asia,45,Male,...,2024-08-24,Yes,No,Linux,Unknown,Debit Card,Spanish,Evening,3,Corporate


In [18]:
date_columns = ["ticket_created_date", "ticket_resolved_date"]

for col in date_columns:
    df[col] = pd.to_datetime(
        df[col],
        format="%Y-%m-%d",   # Change to your format
        errors="coerce"
    )

In [6]:
def check_text_columns(df, columns):
    placeholders = {"test", "n/a", "na", "none", ".", "-", "", "null", "nan"}

    for col in columns:
        s = df[col].fillna("").astype(str)

        print(f"Column: {col}")
        print("Numbers or Punctuation only:", s.str.fullmatch(r"[\d\W_]+", na=False).sum())
        print("Placeholder values:", s.str.strip().str.lower().isin(placeholders).sum())
        print("Whitespace and Newlines:", s.str.contains(r"\n|\r|\t|\s{2,}", regex=True, na=False).sum())
        print("\n")
        
check_text_columns(df, df.select_dtypes(include=['string']).columns)

Column: product
Numbers or Punctuation only: 0
Placeholder values: 0
Whitespace and Newlines: 0


Column: category
Numbers or Punctuation only: 0
Placeholder values: 0
Whitespace and Newlines: 0


Column: issue_description
Numbers or Punctuation only: 0
Placeholder values: 0
Whitespace and Newlines: 0


Column: resolution_notes
Numbers or Punctuation only: 0
Placeholder values: 0
Whitespace and Newlines: 0


Column: priority
Numbers or Punctuation only: 0
Placeholder values: 0
Whitespace and Newlines: 0


Column: status
Numbers or Punctuation only: 0
Placeholder values: 0
Whitespace and Newlines: 0


Column: channel
Numbers or Punctuation only: 0
Placeholder values: 0
Whitespace and Newlines: 0


Column: region
Numbers or Punctuation only: 0
Placeholder values: 0
Whitespace and Newlines: 0


Column: customer_gender
Numbers or Punctuation only: 0
Placeholder values: 0
Whitespace and Newlines: 0


Column: subscription_type
Numbers or Punctuation only: 0
Placeholder values: 0
Whitespace a

In [19]:
def clean_text_columns(df, columns):
    for col in columns:
        df[col] = (
            df[col]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"[^a-z0-9\s\.\?]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

        df = df[df[col] != ""]

    return df.reset_index(drop=True)

df = clean_text_columns(df, df.select_dtypes(include=['string']).columns)

In [8]:
df.head()

,product,category,issue_description,resolution_notes,priority,status,channel,region,customer_age,customer_gender,...,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,web portal,account suspension,the payment was deducted from my bank account ...,data synchronization restored after backend se...,urgent,open,email,north america,31,male,...,2023-05-20,no,yes,macos,edge,paypal,french,afternoon,4,small business
1,mobile app,performance issue,i found a bug in the latest update affecting r...,provided stepbystep troubleshooting instructio...,urgent,closed,email,south america,23,other,...,2024-01-19,yes,yes,windows,firefox,paypal,english,afternoon,2,small business
2,web portal,performance issue,the application crashes whenever i try to uplo...,provided stepbystep troubleshooting instructio...,medium,closed,chat,europe,47,female,...,2022-12-05,yes,yes,windows,safari,bank transfer,french,morning,4,corporate
3,payment gateway,subscription cancellation,my subscription was cancelled without my reque...,provided stepbystep troubleshooting instructio...,medium,closed,social media,asia,65,other,...,2024-04-04,yes,no,windows,chrome,credit card,spanish,afternoon,7,corporate
4,web portal,feature request,the system is not syncing data across devices ...,we have reset the account credentials and advi...,high,pending customer,email,asia,45,male,...,2024-08-24,yes,no,linux,unknown,debit card,spanish,evening,3,corporate


In [20]:
# Parse dates
df['ticket_created_date'] = pd.to_datetime(df['ticket_created_date'])
df['ticket_resolved_date'] = pd.to_datetime(df['ticket_resolved_date'])

# Check future dates
print(df[df['ticket_created_date'] > pd.Timestamp.today()])
print(df[df['ticket_resolved_date'] > pd.Timestamp.today()])


Empty DataFrame
Columns: [product, category, issue_description, resolution_notes, priority, status, channel, region, customer_age, customer_gender, subscription_type, customer_tenure_months, previous_tickets, customer_satisfaction_score, first_response_time_hours, resolution_time_hours, ticket_created_date, ticket_resolved_date, escalated, sla_breached, operating_system, browser, payment_method, language, preferred_contact_time, issue_complexity_score, customer_segment]
Index: []

[0 rows x 27 columns]
Empty DataFrame
Columns: [product, category, issue_description, resolution_notes, priority, status, channel, region, customer_age, customer_gender, subscription_type, customer_tenure_months, previous_tickets, customer_satisfaction_score, first_response_time_hours, resolution_time_hours, ticket_created_date, ticket_resolved_date, escalated, sla_breached, operating_system, browser, payment_method, language, preferred_contact_time, issue_complexity_score, customer_segment]
Index: []

[0 row

In [21]:
# Check created is always before resolved
print(df[df['ticket_created_date'] > df['ticket_resolved_date']])

Empty DataFrame
Columns: [product, category, issue_description, resolution_notes, priority, status, channel, region, customer_age, customer_gender, subscription_type, customer_tenure_months, previous_tickets, customer_satisfaction_score, first_response_time_hours, resolution_time_hours, ticket_created_date, ticket_resolved_date, escalated, sla_breached, operating_system, browser, payment_method, language, preferred_contact_time, issue_complexity_score, customer_segment]
Index: []

[0 rows x 27 columns]


In [22]:
# Validate against existing resolution_time_hours
df['computed_resolution_time'] = (df['ticket_resolved_date'] - df['ticket_created_date']).dt.total_seconds() / 3600
print(df[['resolution_time_hours', 'computed_resolution_time']].describe())

       resolution_time_hours  computed_resolution_time
count          200000.000000              200000.00000
mean              120.539718                 168.20724
std                68.969143                 103.76567
min                 1.000000                   0.00000
25%                60.810000                  72.00000
50%               120.450000                 168.00000
75%               180.260000                 264.00000
max               240.000000                 336.00000


In [28]:
df.drop(columns=["computed_resolution_time"],inplace=True)

In [29]:
import sys
sys.path.append('..')

In [30]:
from src.data.data_loader_saver import save_csv

df.to_csv('../data/processed/cleaned_data.csv', index=False)